# Random forest y RNN estándar con curvas por época

Entrena, con los mismos datos que el experimento final (`models/reports/final/`), dos modelos que faltan en el
gráfico `3_curvas_f1_train_test.png`:

- `classic_features_dictionary`: el **random forest** con atributos + diccionario de capitalización. El RF no entrena
  por épocas, así que **crece por iteraciones** (`warm_start`): en cada una se le agregan árboles hasta llegar a los 100
  del modelo clásico. Con `EPOCHS = 20` son 5 árboles por iteración, la misma cantidad de iteraciones que la Conv + BiLSTM
  conjunta. Con la misma semilla, el bosque final es el mismo que entrenaría el modelo clásico de una sola vez
  (ver `train_classic_model_by_iterations` en `src/training/classic.py`).
- `standard_rnn`: la **RNN estándar** (unidireccional) de cada problema, `EPOCHS` épocas.

El pipeline (`src.pipeline`) los entrena como a cualquier otro experimento, así que cada modelo queda en
`models/runs/<Problema>/<score>__<modelo>/` con su `metadata.json` (historial por época incluido). Al final el notebook
arma un `.zip` con esas carpetas para **descargarlo y descomprimirlo en `models/runs/`** del repo local, y redibuja el
gráfico de curvas con estos modelos en `models/reports/final/poster/` (sin pisar el original).

**En Colab**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU** (la usa la RNN; el RF corre en CPU).

## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BRANCH = "feat/rf-iteraciones-y-rnn"  # rama con el RF por iteraciones y la RNN estándar activada en src/training/train.py
REPO = "/content/TP2-Aprendizaje-Automatico-UBA"

if IN_COLAB:
    if not Path(REPO).exists():
        !git clone https://github.com/SebasAndres/TP2-Aprendizaje-Automatico-UBA.git {REPO}
    %cd {REPO}
    !git fetch origin && git checkout {BRANCH} && git pull
    !pip install -q uv && uv sync
else:
    # Local: el notebook corre desde notebooks/experiments/, subimos hasta la raíz del repo
    %cd ../..

!nvidia-smi -L || echo "Sin GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU"

## 2. Configuración

In [ ]:
EXPERIMENTS = ["classic_features_dictionary", "standard_rnn"]
EPOCHS = 20  # épocas de la RNN e iteraciones del RF (igual que la Conv + BiLSTM conjunta del experimento final)
SIZE = 1_000_000  # máximo de instancias por fuente, igual que el experimento final (~62 mil de train)

# Corridas del experimento final, para comparar los datos y para el gráfico
FINAL_RUNS = ["2026-11-18__04-44-00", "2026-11-18__04-38-20"]
RUNS_DIR = Path("models/runs")

# Las constantes de src/constants.py ya son las del experimento final (class_weight_power 0.25, ventanas de 5 a
# 60 palabras, semilla 12), así que se corre el pipeline tal cual
PIPELINE = "MPLBACKEND=Agg uv run python -m src.pipeline"


def model_folders() -> set[Path]:
    """Carpetas de modelos guardadas: models/runs/<Problema>/<score>__<modelo>/."""
    return {path.parent for path in RUNS_DIR.glob("*/*/metadata.json")}


def metadata(folder: Path) -> dict:
    return json.loads((folder / "metadata.json").read_text(encoding="utf-8"))

## 3. Entrenamiento

El RF arma primero las features de train (~2 millones de palabras) y después agrega árboles; en cada iteración el log
muestra el F1 de train y de test.

In [ ]:
EXPERIMENTS_ARG = ",".join(EXPERIMENTS)
before = model_folders()
!{PIPELINE} --experiments {EXPERIMENTS_ARG} --epochs {EPOCHS} \
    --capitalization-size {SIZE} --punctuation-size {SIZE}

NEW_FOLDERS = sorted(model_folders() - before)
for folder in NEW_FOLDERS:
    print(folder)

## 4. ¿Son los mismos datos que el experimento final?

Los splits dependen solo de la cantidad de instancias y de la semilla: si los tamaños coinciden, son las mismas
instancias.

In [ ]:
final_data = {}
for folder in model_folders():
    meta = metadata(folder)
    if meta["run"] in FINAL_RUNS:
        final_data[meta["problem"]] = meta["data"]

for folder in NEW_FOLDERS:
    meta = metadata(folder)
    same = meta["data"] == final_data.get(meta["problem"])
    print(f"{meta['problem']:22} {meta['model']:28} train={meta['data']['train']['instances']:,} "
          f"test={meta['data']['test']['instances']:,} control={meta['data']['control']['instances']:,} "
          f"{'✓ mismos datos' if same else '✗ DISTINTOS al experimento final'}")

## 5. Descargar las carpetas

El `.zip` tiene la misma estructura que `models/runs/` (`<Problema>/<score>__<modelo>/`). En el repo local:

```bash
unzip runs_rf_rnn.zip -d models/runs/
```

Los `.joblib` del RF pueden pesar bastante (100 árboles sin límite de profundidad sobre ~2 millones de palabras). Para
el gráfico alcanza con `metadata.json`: con `INCLUDE_WEIGHTS = False` el zip lleva solo la metadata y los gráficos.

In [ ]:
import zipfile

INCLUDE_WEIGHTS = True
ZIP_PATH = Path("runs_rf_rnn.zip")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in NEW_FOLDERS:
        for file in sorted(folder.iterdir()):
            if INCLUDE_WEIGHTS or file.suffix not in (".pkl", ".joblib"):
                zf.write(file, file.relative_to(RUNS_DIR))
print(f"{ZIP_PATH}: {ZIP_PATH.stat().st_size / 1e6:.1f} MB")
for name in zipfile.ZipFile(ZIP_PATH).namelist():
    print(" ", name)

if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))

## 6. Curvas de aprendizaje con el RF (y la RNN estándar)

Réplica de `models/reports/final/poster/3_curvas_f1_train_test.png` con los modelos nuevos. Se puede correr en Colab
después de entrenar, o en local después de descomprimir el zip en `models/runs/` (busca la corrida más nueva que tenga
estos modelos; para fijar una, completar `NEW_RUN`).

**Cómo leer el RF**: el eje x es la iteración (5 árboles más cada una), no una época. Su F1 de train queda cerca de 1
porque los árboles crecen hasta el fondo: la brecha con test muestra cuánto sobreajusta.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from notebooks.experiments import candidato_final_plots as C
from src.poster import plots as P
from src.training.report import load_report

%matplotlib inline

NEW_RUN = None  # ej: "2026-11-19__10-00-00"; None = la corrida más nueva con estos modelos
OUT_DIR = Path("models/reports/final/poster")

if NEW_RUN is None:
    runs = sorted(metadata(f)["run"] for f in model_folders() if metadata(f)["model"] in EXPERIMENTS)
    NEW_RUN = runs[-1]
print("Corrida nueva:", NEW_RUN)

folders = [f for f in sorted(model_folders()) if metadata(f)["run"] in FINAL_RUNS + [NEW_RUN]]
report = load_report(folders)
report["config"]["n_epochs"] = f"20 (conjunta), 15 (BiRNN) y {EPOCHS} (RNN) · RF: {EPOCHS} iteraciones de {100 // EPOCHS} árboles"
CAPTION = P.data_caption(report)


def show_and_save(fig, name: str) -> None:
    """Muestra el gráfico y lo guarda en OUT_DIR como PNG y PDF (con nombre nuevo: no pisa el original)."""
    P.add_caption(fig, CAPTION)
    P.save_figure(fig, OUT_DIR, name)
    display(fig)
    plt.close(fig)


show_and_save(
    C.plot_train_test_curves(report, ["joint_conv_bilstm", "bidirectional_rnn", "classic_features_dictionary"]),
    "3_curvas_f1_train_test_rf",
)
show_and_save(
    C.plot_train_test_curves(report, ["joint_conv_bilstm", "bidirectional_rnn", "standard_rnn", "classic_features_dictionary"]),
    "3_curvas_f1_train_test_rf_rnn",
)